# Parser combinators in TypeScript

This notebook is a compact tutorial on **parser combinators** and ends with a working **TypeScript parser** that produces an **abstract syntax tree (AST)**.

The parser supports:

- variables consisting of letters
- numbers consisting of digits
- binary operators `+`, `-`, `*`, `/`, and `^`
- unary functions `exp(...)` and `log(...)`
- parentheses `(` and `)`

The code below is written to avoid the TypeScript narrowing issues you ran into.

## 1. Core parser types

A parser consumes input and returns either:

- a success with a parsed value and the remaining input, or
- a failure with an error message.

The important detail is that `success` uses the literal types `true` and `false`, not `boolean`.

In [1]:
type ParseSuccess<T> = {
  success: true;
  value: T;
  rest: string;
};

type ParseFailure = {
  success: false;
  error: string;
};

type ParseResult<T> = ParseSuccess<T> | ParseFailure;

type Parser<T> = (input: string) => ParseResult<T>;

## 2. Primitive parsers

In [2]:
function char(expected: string): Parser<string> {
  return (input: string): ParseResult<string> => {
    if (input.startsWith(expected)) {
      return {
        success: true,
        value: expected,
        rest: input.slice(expected.length),
      };
    }

    return {
      success: false,
      error: `Expected "${expected}"`,
    };
  };
}

function str(expected: string): Parser<string> {
  return (input: string): ParseResult<string> => {
    if (input.startsWith(expected)) {
      return {
        success: true,
        value: expected,
        rest: input.slice(expected.length),
      };
    }

    return {
      success: false,
      error: `Expected "${expected}"`,
    };
  };
}

function satisfy(
  predicate: (char: string) => boolean,
  errorMessage: string
): Parser<string> {
  return (input: string): ParseResult<string> => {
    const first = input[0];

    if (first !== undefined && predicate(first)) {
      return {
        success: true,
        value: first,
        rest: input.slice(1),
      };
    }

    return {
      success: false,
      error: errorMessage,
    };
  };
}

## 3. Core combinators

These are the standard parser-combinator building blocks.

In [3]:
function map<A, B>(parser: Parser<A>, fn: (value: A) => B): Parser<B> {
  return (input: string): ParseResult<B> => {
    const result = parser(input);

    if ("error" in result) {
      return {
        success: false,
        error: result.error,
      };
    }

    return {
      success: true,
      value: fn(result.value),
      rest: result.rest,
    };
  };
}

function sequence<A, B>(pa: Parser<A>, pb: Parser<B>): Parser<[A, B]> {
  return (input: string): ParseResult<[A, B]> => {
    const ra = pa(input);
    if ("error" in ra) {
      return {
        success: false,
        error: ra.error,
      };
    }

    const rb = pb(ra.rest);
    if ("error" in rb) {
      return {
        success: false,
        error: rb.error,
      };
    }

    return {
      success: true,
      value: [ra.value, rb.value],
      rest: rb.rest,
    };
  };
}

function either<A>(p1: Parser<A>, p2: Parser<A>): Parser<A> {
  return (input: string): ParseResult<A> => {
    const r1 = p1(input);
    if (!("error" in r1)) {
      return r1;
    }

    return p2(input);
  };
}

function many<A>(parser: Parser<A>): Parser<A[]> {
  return (input: string): ParseResult<A[]> => {
    const values: A[] = [];
    let rest = input;

    while (true) {
      const result = parser(rest);
      if ("error" in result) {
        break;
      }

      if (result.rest === rest) {
        return {
          success: false,
          error: "many() received a parser that consumed no input",
        };
      }

      values.push(result.value);
      rest = result.rest;
    }

    return {
      success: true,
      value: values,
      rest,
    };
  };
}

function many1<A>(parser: Parser<A>): Parser<A[]> {
  return (input: string): ParseResult<A[]> => {
    const first = parser(input);
    if ("error" in first) {
      return {
        success: false,
        error: first.error,
      };
    }

    const rest = many(parser)(first.rest);
    if ("error" in rest) {
      return {
        success: false,
        error: rest.error,
      };
    }

    return {
      success: true,
      value: [first.value, ...rest.value],
      rest: rest.rest,
    };
  };
}

function lazy<A>(thunk: () => Parser<A>): Parser<A> {
  return (input: string): ParseResult<A> => thunk()(input);
}

## 4. Whitespace helpers

In [4]:
const letter: Parser<string> = satisfy(
  (c) => /[a-zA-Z]/.test(c),
  "Expected letter"
);

const digit: Parser<string> = satisfy(
  (c) => /[0-9]/.test(c),
  "Expected digit"
);

const whitespace: Parser<string> = satisfy(
  (c) => /\s/.test(c),
  "Expected whitespace"
);

const spaces: Parser<string[]> = many(whitespace);

function token<A>(parser: Parser<A>): Parser<A> {
  return (input: string): ParseResult<A> => {
    const leading = spaces(input);
    if ("error" in leading) {
      return {
        success: false,
        error: leading.error,
      };
    }

    const result = parser(leading.rest);
    if ("error" in result) {
      return {
        success: false,
        error: result.error,
      };
    }

    const trailing = spaces(result.rest);
    if ("error" in trailing) {
      return {
        success: false,
        error: trailing.error,
      };
    }

    return {
      success: true,
      value: result.value,
      rest: trailing.rest,
    };
  };
}

function between<A, B, C>(
  left: Parser<A>,
  middle: Parser<B>,
  right: Parser<C>
): Parser<B> {
  return (input: string): ParseResult<B> => {
    const l = left(input);
    if ("error" in l) {
      return {
        success: false,
        error: l.error,
      };
    }

    const m = middle(l.rest);
    if ("error" in m) {
      return {
        success: false,
        error: m.error,
      };
    }

    const r = right(m.rest);
    if ("error" in r) {
      return {
        success: false,
        error: r.error,
      };
    }

    return {
      success: true,
      value: m.value,
      rest: r.rest,
    };
  };
}

## 5. AST types

In [5]:
type Expr =
  | { type: "Number"; value: number }
  | { type: "Variable"; name: string }
  | { type: "Binary"; op: "+" | "-" | "*" | "/" | "^"; left: Expr; right: Expr }
  | { type: "Call"; fn: "exp" | "log"; arg: Expr };

## 6. Lexical parsers

Variables are made of letters only. Numbers are made of digits only.

In [6]:
const identifier: Parser<string> = map(many1(letter), (chars) => chars.join(""));

const numberLiteral: Parser<number> = map(many1(digit), (chars) =>
  Number(chars.join(""))
);

const numberExpr: Parser<Expr> = map(token(numberLiteral), (value) => ({
  type: "Number",
  value,
}));

const variableExpr: Parser<Expr> = map(token(identifier), (name) => ({
  type: "Variable",
  name,
}));

## 7. Operator helpers

We use:

1. function calls and parentheses
2. exponentiation `^` as right-associative
3. multiplication and division
4. addition and subtraction

In [7]:
function chainLeft<A>(
  valueParser: Parser<A>,
  opParser: Parser<(left: A, right: A) => A>
): Parser<A> {
  return (input: string): ParseResult<A> => {
    const first = valueParser(input);
    if ("error" in first) {
      return {
        success: false,
        error: first.error,
      };
    }

    let acc: A = first.value;
    let rest: string = first.rest;

    while (true) {
      const op = opParser(rest);
      if ("error" in op) {
        break;
      }

      const next = valueParser(op.rest);
      if ("error" in next) {
        return {
          success: false,
          error: "Expected expression after operator",
        };
      }

      acc = op.value(acc, next.value);
      rest = next.rest;
    }

    return {
      success: true,
      value: acc,
      rest,
    };
  };
}

function chainRight<A>(
  valueParser: Parser<A>,
  opParser: Parser<(left: A, right: A) => A>
): Parser<A> {
  function parseFrom(input: string): ParseResult<A> {
    const left = valueParser(input);
    if ("error" in left) {
      return {
        success: false,
        error: left.error,
      };
    }

    const op = opParser(left.rest);
    if ("error" in op) {
      return left;
    }

    const right = parseFrom(op.rest);
    if ("error" in right) {
      return {
        success: false,
        error: right.error,
      };
    }

    return {
      success: true,
      value: op.value(left.value, right.value),
      rest: right.rest,
    };
  }

  return parseFrom;
}

## 8. AST-building operator parsers

In [8]:
const addOp: Parser<(a: Expr, b: Expr) => Expr> = map(token(str("+")), () => {
  return (a, b) => ({ type: "Binary", op: "+", left: a, right: b });
});

const subOp: Parser<(a: Expr, b: Expr) => Expr> = map(token(str("-")), () => {
  return (a, b) => ({ type: "Binary", op: "-", left: a, right: b });
});

const mulOp: Parser<(a: Expr, b: Expr) => Expr> = map(token(str("*")), () => {
  return (a, b) => ({ type: "Binary", op: "*", left: a, right: b });
});

const divOp: Parser<(a: Expr, b: Expr) => Expr> = map(token(str("/")), () => {
  return (a, b) => ({ type: "Binary", op: "/", left: a, right: b });
});

const powOp: Parser<(a: Expr, b: Expr) => Expr> = map(token(str("^")), () => {
  return (a, b) => ({ type: "Binary", op: "^", left: a, right: b });
});

## 9. Grammar

We implement the grammar

```text
expr    := term (("+" | "-") term)*
term    := power (("*" | "/") power)*
power   := primary ("^" power)?
primary := number
         | variable
         | "exp" "(" expr ")"
         | "log" "(" expr ")"
         | "(" expr ")"
```

In [9]:
let expr: Parser<Expr>;

const lparen: Parser<string> = token(str("("));
const rparen: Parser<string> = token(str(")"));

function functionCall(fnName: "exp" | "log"): Parser<Expr> {
  return (input: string): ParseResult<Expr> => {
    const fn = token(str(fnName))(input);
    if ("error" in fn) {
      return {
        success: false,
        error: fn.error,
      };
    }

    const arg = between(lparen, lazy(() => expr), rparen)(fn.rest);
    if ("error" in arg) {
      return {
        success: false,
        error: arg.error,
      };
    }

    return {
      success: true,
      value: {
        type: "Call",
        fn: fnName,
        arg: arg.value,
      },
      rest: arg.rest,
    };
  };
}

const primary: Parser<Expr> = (input: string): ParseResult<Expr> => {
  const parsers: Parser<Expr>[] = [
    functionCall("exp"),
    functionCall("log"),
    between(lparen, lazy(() => expr), rparen),
    numberExpr,
    variableExpr,
  ];

  let lastError = "Expected number, variable, function call, or parenthesized expression";

  for (const p of parsers) {
    const result = p(input);
    if (!("error" in result)) {
      return result;
    }
    lastError = result.error;
  }

  return {
    success: false,
    error: lastError,
  };
};

const power: Parser<Expr> = chainRight(primary, powOp);
const term: Parser<Expr> = chainLeft(power, either(mulOp, divOp));
expr = chainLeft(term, either(addOp, subOp));

[Function (anonymous)]


## 10. Entry point

`parseExpression` checks that the whole input is consumed.

In [10]:
function parseExpression(input: string): ParseResult<Expr> {
  const result = expr(input);

  if ("error" in result) {
    return {
      success: false,
      error: result.error,
    };
  }

  const trailing = spaces(result.rest);
  if ("error" in trailing) {
    return {
      success: false,
      error: trailing.error,
    };
  }

  if (trailing.rest.length > 0) {
    return {
      success: false,
      error: `Unexpected trailing input: "${trailing.rest}"`,
    };
  }

  return {
    success: true,
    value: result.value,
    rest: "",
  };
}

## 11. Example inputs

In [11]:
const examples: string[] = [
  "42",
  "x",
  "a+b*c",
  "(a+b)*c",
  "x^y^z",
  "exp(x)",
  "log(123)",
  "exp(x + 2) / log(y)",
  "a + b * exp(c ^ d) - log(z)"
];

for (const input of examples) {
  console.log("\nINPUT:", input);
  console.log(JSON.stringify(parseExpression(input), null, 2));
}


INPUT: 42
{
  "success": true,
  "value": {
    "type": "Number",
    "value": 42
  },
  "rest": ""
}

INPUT: x
{
  "success": true,
  "value": {
    "type": "Variable",
    "name": "x"
  },
  "rest": ""
}

INPUT: a+b*c
{
  "success": true,
  "value": {
    "type": "Binary",
    "op": "+",
    "left": {
      "type": "Variable",
      "name": "a"
    },
    "right": {
      "type": "Binary",
      "op": "*",
      "left": {
        "type": "Variable",
        "name": "b"
      },
      "right": {
        "type": "Variable",
        "name": "c"
      }
    }
  },
  "rest": ""
}

INPUT: (a+b)*c
{
  "success": true,
  "value": {
    "type": "Binary",
    "op": "*",
    "left": {
      "type": "Binary",
      "op": "+",
      "left": {
        "type": "Variable",
        "name": "a"
      },
      "right": {
        "type": "Variable",
        "name": "b"
      }
    },
    "right": {
      "type": "Variable",
      "name": "c"
    }
  },
  "rest": ""
}

INPUT: x^y^z
{
  "success": true,


## 12. Notes

This parser returns an AST and does not evaluate expressions.

Possible extensions:

- unary minus
- decimal numbers
- source positions
- better error reporting
- AST evaluation